## 🏗️ Bronze Layer - Raw Data Generation

### Purpose
Generate **synthetic CRM campaign data** using the Faker library for demonstration and testing purposes. This notebook creates realistic raw data that simulates a CRM system tracking marketing campaigns, customer communications, and resulting sales.

---

### Tables Created

All bronze tables are written to Unity Catalog in the `workspace.crm_info` schema with the prefix `bronze_*`:

* **`bronze_customers`** - 1,000 customer records
  - Segments: Premium, Standard, Basic
  - Regions: North, South, East, West, Central
  - Age groups: 18-24, 25-34, 35-44, 45-54, 55-64, 65+
  - Lifetime value range: $100 - $50,000

* **`bronze_products`** - 15 financial products across 5 categories
  - Categories: Banking, Credit Cards, Loans, Investment, Insurance
  - Price ranges: $0 (free) to $300,000 (mortgages)

* **`bronze_campaigns`** - 60 campaigns (15 products × 4 channels)
  - Channels: Email, SMS, Call, Push Notification
  - Types: Acquisition, Retention, Cross-sell, Win-back
  - Budgets: $5K - $50K per campaign
  - Duration: 7-30 days within the 60-day window

* **`bronze_communications`** - ~5,600 communication events
  - Volume: Each customer receives 3-8 communications over 60 days
  - Engagement tracking: delivered, opened, clicked
  - Channel-specific engagement rates based on industry benchmarks

* **`bronze_sales`** - ~1,100 sales (20% conversion rate)
  - Attribution window: 15 days from communication to sale
  - Conversion lag: 0-15 days, with most sales occurring within 7 days

---

### Data Characteristics

**Time Range**: 60-day rolling window (2 months ending today)
* Start date: Today - 60 days
* End date: Today
* Campaign dates: Randomly distributed within this window

**Channel-Specific Engagement Rates** (Modeled after industry benchmarks):
* **Email**: 98% delivery, 35% open, 15% click
* **SMS**: 99% delivery, 90% open, 18% click
* **Call**: 100% delivery, 100% open, 45% engagement
* **Push Notification**: 97% delivery, 25% open, 8% click

**Segment-Based Channel Preferences**:
* **Premium**: Prefers Call (40%) and Email (40%)
* **Standard**: Balanced across channels, slight Email preference (50%)
* **Basic**: Heavily prefers Email (60%) and SMS (30%)

**Reproducibility**: 
* Fixed random seeds (`Faker.seed(42)`, `random.seed(42)`)
* Running this notebook multiple times produces identical data

---

### Quality Attributes

All bronze tables include standard **audit columns**:
* `created_at` - Timestamp when record was created
* `updated_at` - Timestamp of last update
* `is_active` - Boolean flag for soft deletes
* `source_system` - Set to 'synthetic_crm' for all records

---

### Next Steps

After generating bronze data:
1. **Silver Layer** (`CRM_silver_star_schema`) reads bronze tables and creates:
   - Dimension tables (date, customer, product, campaign, channel)
   - Fact table (fact_crm_communication) with engagement metrics
   - Forecasting models (Prophet for revenue/conversions)

2. **Gold Layer** (`CRM_gold_analytics_dashboard`) creates:
   - Pre-aggregated tables for dashboard performance
   - Advanced analytics (cohorts, LTV analysis, fatigue analysis)

In [0]:
%pip install faker
%pip install prophet --quiet
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from faker import Faker
import random
from datetime import datetime, timedelta
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import *

import time

fake = Faker()
# Use timestamp as seed to get different but reproducible data each run
seed = int(time.time())
Faker.seed(seed)
random.seed(seed)
print(f"Using seed: {seed}")

print("Libraries imported successfully!")

Libraries imported successfully!


In [0]:
# Generate Customers (1000 records)
from pyspark.sql.functions import current_timestamp, lit

age_groups = ['18-24', '25-34', '35-44', '45-54', '55-64', '65+']
segments = ['Premium', 'Standard', 'Basic']
regions = ['North', 'South', 'East', 'West', 'Central']

customers = []
for i in range(1000):
    customer = {
        'customer_id': i + 1,
        'first_name': fake.first_name(),
        'last_name': fake.last_name(),
        'email': fake.email(),
        'phone': fake.phone_number(),
        'age_group': random.choice(age_groups),
        'segment': random.choice(segments),
        'region': random.choice(regions),
        'customer_since': fake.date_between(start_date='-3y', end_date='-2m'),
        'lifetime_value': round(random.uniform(100, 50000), 2)
    }
    customers.append(customer)

df_customers = spark.createDataFrame(pd.DataFrame(customers))

# Add audit columns
df_customers = df_customers.withColumn('created_at', current_timestamp()) \
                           .withColumn('updated_at', current_timestamp()) \
                           .withColumn('is_active', lit(True)) \
                           .withColumn('source_system', lit('synthetic_crm'))
print(f"Generated {len(customers)} customers")
print(f"\nSegment distribution:")
display(df_customers.groupBy('segment').count().orderBy('segment'))

Generated 1000 customers

Segment distribution:


segment,count
Basic,317
Premium,327
Standard,356


In [0]:
display(df_customers.head(20))

customer_id,first_name,last_name,email,phone,age_group,segment,region,customer_since,lifetime_value,created_at,updated_at,is_active,source_system
1,Danielle,Johnson,john21@example.net,001-581-896-0013x3890,65+,Premium,North,2024-03-07,37103.37,2026-07-10T12:55:46.985Z,2026-07-10T12:55:46.985Z,true,synthetic_crm
2,Bridget,Pacheco,blakeerik@example.com,942-335-1161x55940,25-34,Premium,South,2026-06-09,36849.91,2026-07-10T12:55:46.985Z,2026-07-10T12:55:46.985Z,true,synthetic_crm
3,Nicholas,Arnold,barbara10@example.net,441.731.6475,65+,Basic,Central,2024-08-02,4438.25,2026-07-10T12:55:46.985Z,2026-07-10T12:55:46.985Z,true,synthetic_crm
4,Elizabeth,Miles,lynchgeorge@example.net,527.264.8350,45-54,Premium,North,2025-12-07,4775.39,2026-07-10T12:55:46.985Z,2026-07-10T12:55:46.985Z,true,synthetic_crm
5,Richard,Jones,jason76@example.net,724.523.8849x696,25-34,Basic,Central,2026-07-06,1424.14,2026-07-10T12:55:46.985Z,2026-07-10T12:55:46.985Z,true,synthetic_crm
6,Devon,Snyder,callahaneric@example.org,+1-669-716-6978x48018,25-34,Basic,Central,2023-11-09,21034.04,2026-07-10T12:55:46.985Z,2026-07-10T12:55:46.985Z,true,synthetic_crm
7,Ryan,Chavez,teresa28@example.org,+1-948-293-2528x8095,45-54,Basic,East,2023-11-09,40490.58,2026-07-10T12:55:46.985Z,2026-07-10T12:55:46.985Z,true,synthetic_crm
8,Michael,Galloway,john39@example.org,671-882-2782,18-24,Premium,West,2026-05-30,17078.5,2026-07-10T12:55:46.985Z,2026-07-10T12:55:46.985Z,true,synthetic_crm
9,Sarah,Campos,jenniferross@example.net,+1-557-587-1331x5098,25-34,Premium,East,2023-07-17,5200.29,2026-07-10T12:55:46.985Z,2026-07-10T12:55:46.985Z,true,synthetic_crm
10,Jean,Brown,whiteheadmichele@example.org,334.373.8299x73763,45-54,Premium,East,2025-06-30,42389.97,2026-07-10T12:55:46.985Z,2026-07-10T12:55:46.985Z,true,synthetic_crm


In [0]:
# Generate Products (15 products across different categories)
from pyspark.sql.functions import current_timestamp, lit

product_data = [
    {'category': 'Banking', 'product_name': 'Savings Account', 'price': 0},
    {'category': 'Banking', 'product_name': 'Premium Checking', 'price': 15.99},
    {'category': 'Banking', 'product_name': 'Business Account', 'price': 29.99},
    {'category': 'Credit Cards', 'product_name': 'Cashback Card', 'price': 0},
    {'category': 'Credit Cards', 'product_name': 'Travel Rewards Card', 'price': 95},
    {'category': 'Credit Cards', 'product_name': 'Premium Card', 'price': 450},
    {'category': 'Loans', 'product_name': 'Personal Loan', 'price': 5000},
    {'category': 'Loans', 'product_name': 'Auto Loan', 'price': 25000},
    {'category': 'Loans', 'product_name': 'Home Mortgage', 'price': 300000},
    {'category': 'Investment', 'product_name': 'Index Fund', 'price': 100},
    {'category': 'Investment', 'product_name': 'Managed Portfolio', 'price': 10000},
    {'category': 'Investment', 'product_name': 'Retirement Account', 'price': 5000},
    {'category': 'Insurance', 'product_name': 'Life Insurance', 'price': 50},
    {'category': 'Insurance', 'product_name': 'Auto Insurance', 'price': 120},
    {'category': 'Insurance', 'product_name': 'Home Insurance', 'price': 150}
]

products = []
for i, prod in enumerate(product_data):
    product = {
        'product_id': i + 1,
        'product_name': prod['product_name'],
        'category': prod['category'],
        'base_price': prod['price']
    }
    products.append(product)

df_products = spark.createDataFrame(pd.DataFrame(products))

# Add audit columns
df_products = df_products.withColumn('created_at', current_timestamp()) \
                         .withColumn('updated_at', current_timestamp()) \
                         .withColumn('is_active', lit(True)) \
                         .withColumn('source_system', lit('synthetic_crm'))
print(f"Generated {len(products)} products")
print(f"\nProduct category distribution:")
display(df_products.groupBy('category').count().orderBy('category'))

Generated 15 products

Product category distribution:


category,count
Banking,3
Credit Cards,3
Insurance,3
Investment,3
Loans,3


In [0]:
# Generate Campaigns
from pyspark.sql.functions import current_timestamp, lit

channels = ['Email', 'SMS', 'Call', 'Push Notification']
channel_map = {'Email': 1, 'SMS': 2, 'Call': 3, 'Push Notification': 4}
campaign_types = ['Acquisition', 'Retention', 'Cross-sell', 'Win-back']
campaign_goals = ['Awareness', 'Engagement', 'Conversion', 'Loyalty']
campaign_status = ['Active', 'Paused', 'Completed']
budget_ranges = [5000, 10000, 15000, 25000, 50000]

# Define 2-month period for campaigns
end_date = datetime.now().date()
start_date = end_date - timedelta(days=60)

campaigns = []
campaign_id = 1

for product in products:
    for channel in channels:
        campaign_start = start_date + timedelta(days=random.randint(0, 45))
        campaign_end = min(campaign_start + timedelta(days=random.randint(7, 30)), end_date)
        campaign = {
            'campaign_id': campaign_id,
            'campaign_name': f"{product['product_name']} via {channel}",
            'product_id': product['product_id'],
            'channel': channel,
            'channel_id': channel_map[channel],
            'campaign_type': random.choice(campaign_types),
            'campaign_goal': random.choice(campaign_goals),
            'campaign_status': random.choices(campaign_status, weights=[0.7, 0.1, 0.2])[0],
            'budget': random.choice(budget_ranges),
            'start_date': campaign_start,
            'end_date': campaign_end,
            'campaign_duration_days': abs((campaign_end - campaign_start).days)
        }
        campaigns.append(campaign)
        campaign_id += 1

df_campaigns = spark.createDataFrame(pd.DataFrame(campaigns))

# Add audit columns
df_campaigns = df_campaigns.withColumn('created_at', current_timestamp()) \
                           .withColumn('updated_at', current_timestamp()) \
                           .withColumn('is_active', lit(True)) \
                           .withColumn('source_system', lit('synthetic_crm'))
print(f"Generated {len(campaigns)} campaigns (15 products × 4 channels)")
print(f"Campaign period: {start_date} to {end_date}")
display(df_campaigns.limit(10))

Generated 60 campaigns (15 products × 4 channels)
Campaign period: 2026-05-11 to 2026-07-10


campaign_id,campaign_name,product_id,channel,channel_id,campaign_type,campaign_goal,campaign_status,budget,start_date,end_date,campaign_duration_days,created_at,updated_at,is_active,source_system
1,Savings Account via Email,1,Email,1,Cross-sell,Conversion,Active,25000,2026-06-03,2026-06-10,7,2026-07-10T12:55:52.603Z,2026-07-10T12:55:52.603Z,true,synthetic_crm
2,Savings Account via SMS,1,SMS,2,Acquisition,Engagement,Active,25000,2026-06-24,2026-07-10,16,2026-07-10T12:55:52.603Z,2026-07-10T12:55:52.603Z,true,synthetic_crm
3,Savings Account via Call,1,Call,3,Win-back,Awareness,Active,10000,2026-05-31,2026-06-15,15,2026-07-10T12:55:52.603Z,2026-07-10T12:55:52.603Z,true,synthetic_crm
4,Savings Account via Push Notification,1,Push Notification,4,Win-back,Conversion,Completed,50000,2026-06-06,2026-07-06,30,2026-07-10T12:55:52.603Z,2026-07-10T12:55:52.603Z,true,synthetic_crm
5,Premium Checking via Email,2,Email,1,Cross-sell,Loyalty,Active,50000,2026-05-21,2026-05-30,9,2026-07-10T12:55:52.603Z,2026-07-10T12:55:52.603Z,true,synthetic_crm
6,Premium Checking via SMS,2,SMS,2,Retention,Engagement,Completed,10000,2026-06-16,2026-06-23,7,2026-07-10T12:55:52.603Z,2026-07-10T12:55:52.603Z,true,synthetic_crm
7,Premium Checking via Call,2,Call,3,Acquisition,Loyalty,Active,15000,2026-05-14,2026-05-28,14,2026-07-10T12:55:52.603Z,2026-07-10T12:55:52.603Z,true,synthetic_crm
8,Premium Checking via Push Notification,2,Push Notification,4,Cross-sell,Conversion,Completed,50000,2026-06-25,2026-07-08,13,2026-07-10T12:55:52.603Z,2026-07-10T12:55:52.603Z,true,synthetic_crm
9,Business Account via Email,3,Email,1,Acquisition,Awareness,Active,25000,2026-06-05,2026-07-02,27,2026-07-10T12:55:52.603Z,2026-07-10T12:55:52.603Z,true,synthetic_crm
10,Business Account via SMS,3,SMS,2,Retention,Loyalty,Active,15000,2026-06-18,2026-07-09,21,2026-07-10T12:55:52.603Z,2026-07-10T12:55:52.603Z,true,synthetic_crm


In [0]:
# Generate CRM Communications (2-month daily history)
channel_map = {'Email': 1, 'SMS': 2, 'Call': 3, 'Push Notification': 4}

communications = []
comm_id = 1

# Define 2-month period ending today
end_date = datetime.now().date()
start_date = end_date - timedelta(days=60)  # 2 months = 60 days

# Each customer receives 3-8 communications over the 2-month period
for customer in customers:
    num_comms = random.randint(3, 8)
    
    # Generate random dates for this customer's communications
    for _ in range(num_comms):
        
        # Channel preferences by segment
        if customer['segment'] == 'Premium':
            channel = random.choices(channels, weights=[0.4, 0.1, 0.4, 0.1])[0]
        elif customer['segment'] == 'Standard':
            channel = random.choices(channels, weights=[0.5, 0.2, 0.2, 0.1])[0]
        else:  # Basic
            channel = random.choices(channels, weights=[0.6, 0.3, 0.05, 0.05])[0]

        # Select random campaign and communication date
        available_campaigns = [c for c in campaigns if c['channel'] == channel]
        campaign = random.choice(available_campaigns)
        comm_date = fake.date_between(start_date=campaign['start_date'], end_date=campaign['end_date'])
        
        if channel == "Email":
            delivered = random.random() < 0.98
            opened = delivered and random.random() < 0.35
            clicked = opened and random.random() < 0.15
        elif channel == "SMS":
            delivered = random.random() < 0.99
            opened = delivered and random.random() < 0.90
            clicked = opened and random.random() < 0.18
        elif channel == 'Push Notification':
            delivered = random.random() < 0.97
            opened = delivered and random.random() < 0.25
            clicked = opened and random.random() < 0.08
        else:  # Call
            delivered = True
            opened = True
            clicked = opened and random.random() < 0.45

        communication = {
            'communication_id': comm_id,
            'customer_id': customer['customer_id'],
            'campaign_id': campaign['campaign_id'],
            'product_id': campaign['product_id'],
            'channel': channel,
            'channel_id': channel_map[channel],
            'delivered': delivered,
            'opened': opened,
            'clicked': clicked,
            'communication_date': comm_date
        }
        communications.append(communication)
        comm_id += 1

df_communications = spark.createDataFrame(pd.DataFrame(communications))
print(f"Generated {len(communications)} CRM communications over 2 months")
print(f"\nDate range: {start_date} to {end_date}")
print(f"\nChannel distribution:")
display(df_communications.groupBy('channel').count().orderBy('channel'))

Generated 5598 CRM communications over 2 months

Date range: 2026-05-11 to 2026-07-10

Channel distribution:


channel,count
Call,1253
Email,2776
Push Notification,482
SMS,1087


In [0]:
display(df_communications.limit(20))

communication_id,customer_id,campaign_id,product_id,channel,channel_id,delivered,opened,clicked,communication_date
1,1,25,7,Email,1,true,false,false,2026-07-04
2,1,59,15,Call,3,true,true,false,2026-06-03
3,1,12,3,Push Notification,4,true,false,false,2026-05-25
4,1,31,8,Call,3,true,true,true,2026-05-28
5,1,21,6,Email,1,true,false,false,2026-06-08
6,1,47,12,Call,3,true,true,false,2026-06-08
7,2,41,11,Email,1,true,false,false,2026-06-11
8,2,39,10,Call,3,true,true,true,2026-06-25
9,2,15,4,Call,3,true,true,false,2026-06-09
10,2,43,11,Call,3,true,true,true,2026-07-06


In [0]:
# Generate Sales with 15-day attribution window
# A sale is attributed to a CRM communication if it occurs within 15 days of the message

sales = []
sale_id = 1
attribution_window_days = 15

# Sort communications by customer and date for easier processing
comms_sorted = sorted(communications, key=lambda x: (x['customer_id'], x['communication_date']))

# Generate sales - approximately 20% conversion rate from communications
for comm in communications:
    # 20% chance this communication results in a sale
    if random.random() < 0.20:
        # Sale happens within 15 days of the communication
        days_until_sale = random.randint(0, attribution_window_days)
        sale_date = comm['communication_date'] + timedelta(days=days_until_sale)
        
        # Don't create sales in the future
        if sale_date <= end_date:
            # Get product details
            product = next(p for p in products if p['product_id'] == comm['product_id'])
            
            # Calculate sale amount with some variation
            base_price = product['base_price']
            if base_price == 0:
                sale_amount = random.uniform(100, 500)  # For free products, use a service value
            else:
                sale_amount = base_price * random.uniform(0.9, 1.1)  # ±10% variation
            
            sale = {
                'sale_id': sale_id,
                'customer_id': comm['customer_id'],
                'product_id': comm['product_id'],
                'communication_id': comm['communication_id'],  # Attribution link
                'sale_date': sale_date,
                'sale_amount': round(sale_amount, 2),
                'days_from_communication': days_until_sale
            }
            sales.append(sale)
            sale_id += 1

df_sales = spark.createDataFrame(pd.DataFrame(sales))
print(f"Generated {len(sales)} sales")
print(f"\nConversion rate: {len(sales) / len(communications) * 100:.1f}%")
print(f"\nAttribution window analysis:")
display(df_sales.groupBy('days_from_communication').count().orderBy('days_from_communication').limit(16))

Generated 982 sales

Conversion rate: 17.5%

Attribution window analysis:


days_from_communication,count
0,61
1,65
2,72
3,64
4,69
5,63
6,69
7,65
8,60
9,52


In [0]:
display(df_sales.limit(20))

sale_id,customer_id,product_id,communication_id,sale_date,sale_amount,days_from_communication
1,1,12,6,2026-06-21,5195.76,13
2,2,14,13,2026-07-03,124.86,5
3,4,14,22,2026-06-18,113.18,6
4,4,6,24,2026-07-10,445.75,13
5,5,5,26,2026-07-09,97.98,15
6,5,7,30,2026-06-27,5150.46,4
7,6,15,34,2026-06-14,159.32,7
8,6,15,36,2026-05-28,148.02,7
9,6,6,38,2026-06-09,463.39,3
10,7,1,47,2026-06-14,197.86,8


In [0]:
# ⚠️ DEPRECATED - For reference only. Use fact_crm_communication (proper star schema) instead.
# This denormalized table was created before the star schema design.
#
# Create Dashboard Fact Table with Daily, Weekly, Monthly aggregations
from pyspark.sql.functions import col, date_format, weekofyear, year, month, weekofyear, dayofmonth, count as spark_count, sum as spark_sum, countDistinct, concat, lit, lpad, coalesce, when

# Join communications with products to get category and product name
df_comm_products = df_communications.join(
    df_products,
    df_communications.product_id == df_products.product_id,
    'left'
).join(
    df_campaigns,
    df_communications.campaign_id == df_campaigns.campaign_id,
    'left'
).select(
    df_communications.communication_id,
    df_communications.customer_id,
    df_communications.campaign_id,
    df_communications.product_id,
    df_products.product_name,
    df_products.category,
    df_communications.channel,
    df_communications.delivered,
    df_communications.opened,
    df_communications.clicked,
    df_communications.communication_date
)

# Join with sales to get conversion information
df_comm_with_sales = df_comm_products.join(
    df_sales.select('communication_id', 'sale_id', 'sale_amount'),
    'communication_id',
    'left'
)

# Add time dimensions
df_comm_with_sales = df_comm_with_sales.withColumn('date', col('communication_date')) \
    .withColumn('year', year('communication_date')) \
    .withColumn('month', month('communication_date')) \
    .withColumn('week', weekofyear('communication_date')) \
    .withColumn('day', dayofmonth('communication_date')) \
    .withColumn('year_month', date_format('communication_date', 'yyyy-MM')) \
    .withColumn('year_week', concat(col('year').cast('string'), lit('-W'), lpad(col('week').cast('string'), 2, '0'))) \
    .withColumn('sale_amount', coalesce(col('sale_amount'), lit(0))) \
    .withColumn('converted', when(col('sale_id').isNotNull(), 1).otherwise(0))

# Calendar daily aggregation
calendar = (
    df_comm_with_sales.select('date', 'year', 'month', 'week', 'year_month', 'year_week', 'day')
        .withColumn('time_grain', lit('Daily'))
)

# Create aggregated fact table
df_fact_table = df_comm_with_sales

print(f"Generated fact table with {df_fact_table.count()} records")
print(f"\nSample aggregation - Communications and Sales by Product:")
display(
    df_fact_table.groupBy('product_name', 'category') \
        .agg(
            countDistinct('customer_id').alias('unique_customers'),
            spark_count('communication_id').alias('total_communications'),
            spark_count('sale_id').alias('total_sales'),
            spark_sum('sale_amount').alias('total_revenue')
        ) \
        .orderBy('total_sales', ascending=False)
)

Generated fact table with 5598 records

Sample aggregation - Communications and Sales by Product:


product_name,category,unique_customers,total_communications,total_sales,total_revenue
Premium Checking,Banking,315,364,75,1198.4599999999996
Index Fund,Investment,320,377,73,7294.959999999999
Life Insurance,Insurance,324,387,73,3629.480000000001
Auto Insurance,Insurance,327,384,69,8212.34
Cashback Card,Credit Cards,322,363,69,21221.560000000005
Auto Loan,Loans,312,350,68,1699246.6199999985
Business Account,Banking,338,417,68,2014.8600000000006
Home Insurance,Insurance,306,361,67,10008.060000000003
Personal Loan,Loans,325,390,66,329326.1
Savings Account,Banking,330,405,65,20685.0
